In [1]:
try:
    import transformers
    import datasets
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
except:
    print("all or at least one of the package not installed. Installing now. Please remember to restart kernel once done")
    !pip install transformers bitsandbytes accelerate datasets peft

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch
from torch.utils.data import DataLoader, Dataset
from functools import partial
from peft import AutoPeftModelForCausalLM

In [3]:
#importing the "King" library :P
import sagemaker
import boto3
from sagemaker.inputs import TrainingInput

#sagemaker pytorch container estimator related libraries
from sagemaker.pytorch import PyTorch
from sagemaker.pytorch import PyTorchModel


from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.predictor import Predictor

#general utility imports
import os
import glob
import pandas as pd
import numpy as np
import random

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/06/25 08:11:30] INFO     Found credentials from IAM Role:                                   ]8;id=910595;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=197134;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [16]:
ds = load_dataset("Abirate/english_quotes")

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [17]:
ds

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [18]:
ds['train'][:10]

{'quote': ['“Be yourself; everyone else is already taken.”',
  "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”",
  "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”",
  '“So many books, so little time.”',
  '“A room without books is like a body without a soul.”',
  "“Be who you are and say what you feel, because those who mind don't matter, and those who matter don't mind.”",
  "“You've gotta dance like there's nobody watching,Love like you'll never be hurt,Sing like there's nobody listening,And live like it's heaven on earth.”",
  "“You know you're in love when you can't fall asleep because reality is finally better than your dreams.”",
  '“You only live once, but if you do it right, once is enough.”',
  '“Be the change that you wish to see in the world.”'],
 'author': ['Oscar Wild

In [19]:
df = ds['train'].to_pandas()

In [20]:
df

,quote,author,tags
0,“Be yourself; everyone else is already taken.”,Oscar Wilde,"[be-yourself, gilbert-perreira, honesty, inspi..."
1,"“I'm selfish, impatient and a little insecure....",Marilyn Monroe,"[best, life, love, mistakes, out-of-control, t..."
2,“Two things are infinite: the universe and hum...,Albert Einstein,"[human-nature, humor, infinity, philosophy, sc..."
3,"“So many books, so little time.”",Frank Zappa,"[books, humor]"
4,“A room without books is like a body without a...,Marcus Tullius Cicero,"[books, simile, soul]"
...,...,...,...
2503,“Morality is simply the attitude we adopt towa...,"Oscar Wilde,","[morality, philosophy]"
2504,“Don't aim at success. The more you aim at it ...,"Viktor E. Frankl,","[happiness, success]"
2505,"“In life, finding a voice is speaking and livi...",John Grisham,[inspirational-life]
2506,"“Winter is the time for comfort, for good food...",Edith Sitwell,"[comfort, home, winter]"


In [22]:
df.author.value_counts()

author
Cassandra Clare,               99
J.K. Rowling,                  74
John Green,                    53
Roy T. Bennett,                46
Mark Twain                     42
                               ..
Thich Nhat Hanh,                1
Bertrand Russell,               1
Ø£Ø­Ù„Ø§Ù… Ù…Ø³ØªØºØ§Ù†Ù…ÙŠ     1
James Patterson,                1
G.K. Chesterton,                1
Name: count, Length: 880, dtype: int64

In [23]:
class dset(Dataset):
    def __init__(self, ds, start_prompt, end_prompt):
        super().__init__()
        self.data = ds
        self.start_prompt = start_prompt
        self.end_prompt = end_prompt

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        author = self.data.iloc[idx]['author']
        quote = self.data.iloc[idx]['quote']
        x = self.start_prompt + author
        y = self.end_prompt + quote
        return x, y

In [24]:
start_prompt = f'Please generate a quote written by author name given below:\n\nauthor: '
end_prompt = f'quote: '
print(start_prompt)
print("#########################")
print(end_prompt)

Please generate a quote written by author name given below:

author: 
#########################
quote: 


In [25]:
train_ds = dset(df, start_prompt, end_prompt)

In [31]:
print(f'{train_ds[3][0]}\n\n')
print(train_ds[3][1])

Please generate a quote written by author name given below:

author: Frank Zappa


quote: “So many books, so little time.”


In [54]:
ckpt = 'facebook/opt-350m'
tokenizer = AutoTokenizer.from_pretrained(ckpt)
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [93]:
def collate_fn(batch, tokenizer):
    eos_tokenid = [tokenizer.vocab[tokenizer.eos_token]]
    pad_tokenid = [tokenizer.vocab[tokenizer.pad_token]]
    ignore_loss_id = [-100]
    ignore_token_id = [0]
    inp_list, att_list, y_list = [],  [], []

    for x, y in batch:
        x_mod = x + "\n"  # f'{x}\n\n'
        print(x_mod)
        x_tok = tokenizer(x_mod, truncation=True).input_ids
        y_tok = tokenizer(y, truncation=True).input_ids
        print(f'X: {x_tok}\nY: {y_tok}')

        prompt = x_mod + y
        input_prompt = tokenizer(prompt, truncation=True)

        x_tok_len = len(x_tok)
        label_prompt = ignore_loss_id*(x_tok_len - 1) + y_tok[1:] + eos_tokenid
        #to remove extra start of sentence token from y_tok

        inp_list.append(input_prompt['input_ids'])
        att_list.append(input_prompt['attention_mask'])
        print(f"{len(input_prompt['input_ids'])}\n{len(input_prompt['attention_mask'])}\n{input_prompt}")

        y_list.append(label_prompt)
        print(f'{len(label_prompt)}\n{label_prompt}')

    len_labs = [len(l) for l in y_list]
    max_len = max(len_labs)

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    #print(inp_list, max_len, pad_tokenid)
    input_ids = pad_tokens_stack(inp_list, max_len, pad_tokenid)
    attention_mask = pad_tokens_stack(att_list, max_len, ignore_token_id)
    labels = pad_tokens_stack(y_list, max_len, ignore_loss_id)

    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [94]:
cpu_cores = os.cpu_count()
wrapper_collate_fn = partial(
        collate_fn,
        tokenizer=tokenizer
        )
train_dl = DataLoader(train_ds, shuffle=False, batch_size=3,
                      num_workers=cpu_cores, collate_fn=wrapper_collate_fn)

In [96]:
iter_dl = iter(train_dl)

Please generate a quote written by author name given below:

author: Frank Zappa



Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


X: [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 3848, 525, 22181, 50118]
Y: [2, 43948, 35, 44, 48, 2847, 171, 2799, 6, 98, 410, 86, 4, 17, 46]
34
34
{'input_ids': [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 3848, 525, 22181, 50118, 43948, 35, 44, 48, 2847, 171, 2799, 6, 98, 410, 86, 4, 17, 46], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
34
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 43948, 35, 44, 48, 2847, 171, 2799, 6, 98, 410, 86, 4, 17, 46, 2]
Please generate a quote written by author name given below:

author: Marcus Tullius Cicero

X: [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 7380, 255, 5023, 6125, 31547, 5160, 50118]
Y: [2, 43948, 35, 44, 48, 250, 929, 396, 2799, 16, 101, 10, 809, 396, 10, 7047, 4, 17, 46]


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


X: [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 5887, 33941, 50118]
Y: [2, 43948, 35, 44, 48, 9325, 2512, 131, 961, 1493, 16, 416, 551, 4, 17, 46]
34
34
{'input_ids': [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 5887, 33941, 50118, 43948, 35, 44, 48, 9325, 2512, 131, 961, 1493, 16, 416, 551, 4, 17, 46], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
34
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 43948, 35, 44, 48, 9325, 2512, 131, 961, 1493, 16, 416, 551, 4, 17, 46, 2]
Please generate a quote written by author name given below:

author: Marilyn Monroe

X: [2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 23542, 12885, 50118]
Y: [2, 43948, 35, 44, 48, 100, 437, 20462, 6, 32601, 8, 10, 410, 27810, 4, 38, 146, 6160, 6, 38, 524, 66, 9, 

In [83]:
print(tokenizer.decode([2, 6715, 5368, 10, 9740, 1982, 30, 2730, 766, 576, 874, 35, 50118, 50118, 11515, 35, 5887, 33941, 50140]))

</s>Please generate a quote written by author name given below:

author: Oscar Wilde




In [85]:
print(tokenizer.decode([2, 43948, 35, 44, 48, 9325, 2512, 131, 961, 1493, 16, 416, 551, 4, 17, 46]))

</s>quote: “Be yourself; everyone else is already taken.”


In [86]:
tokenizer.decode(50140)

'\n\n'

In [68]:
tokenizer.decode([43948, 35, 44, 48, 9325, 2512, 131, 961, 1493, 16, 416, 551, 4, 17, 46, 2])

'quote: “Be yourself; everyone else is already taken.”</s>'

In [70]:
tokenizer.decode([43948, 35, 44, 48, 100, 437, 20462, 6, 32601, 8, 10, 410, 27810, 4, 38, 146, 6160, 6, 38, 524, 66, 9, 797, 8, 23, 498, 543, 7, 3679, 4, 125, 114, 47, 64, 75, 3679, 162, 23, 127, 2373, 6, 172, 47, 686, 25, 7105, 218, 75, 6565, 162, 23, 127, 275, 4, 17, 46, 2])

"quote: “I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”</s>"

In [67]:
x, y = next(iter_dl)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 x, y = next(iter_dl)                                                                         │
│   2                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/data/dataloa │
│ der.py:631 in __next__                                                                           │
│                                                                                                  │
│    628 │   │   │   if self._sampler_iter is None:                                                │
│    629 │   │   │   │   # TODO(https://github.com/pytorch/pytorch/issues/76750)                   │
│    630 │   │   │   │   self._reset()  # type: ignore[call-arg]                                   │
│ ❱  631 │   │   │   data = self._next_data()                                                      │
│    632 │   │   │   self._num_yielded += 1                                                        │
│    633 │   │   │   if self._dataset_kind == _DatasetKind.Iterable and \                          │
│    634 │   │   │   │   │   self._IterableDataset_len_called is not None and \                    │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/data/dataloa │
│ der.py:1346 in _next_data                                                                        │
│                                                                                                  │
│   1343 │   │   │   │   self._task_info[idx] += (data,)                                           │
│   1344 │   │   │   else:                                                                         │
│   1345 │   │   │   │   del self._task_info[idx]                                                  │
│ ❱ 1346 │   │   │   │   return self._process_data(data)                                           │
│   1347 │                                                                                         │
│   1348 │   def _try_put_index(self):                                                             │
│   1349 │   │   assert self._tasks_outstanding < self._prefetch_factor * self._num_workers        │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/data/dataloa │
│ der.py:1372 in _process_data                                                                     │
│                                                                                                  │
│   1369 │   │   self._rcvd_idx += 1                                                               │
│   1370 │   │   self._try_put_index()                                                             │
│   1371 │   │   if isinstance(data, ExceptionWrapper):                                            │
│ ❱ 1372 │   │   │   data.reraise()                                                                │
│   1373 │   │   return data                                                                       │
│   1374 │                                                                                         │
│   1375 │   def _mark_worker_as_unavailable(self, worker_id, shutdown=False):                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/s

In [10]:
df['full_text'] = df.apply(lambda x: f'author: {x["author"]}\nquote:{x["quote"]}', axis=1)

In [11]:
print(df['full_text'].iloc[9])

author: Mahatma Gandhi
quote:“Be the change that you wish to see in the world.”


In [12]:
start_prompt = f'Please generate a quote written by author name given below:\n\nauthor: '
end_prompt = f'\n\nquote: '
print(start_prompt, end_prompt)

Please generate a quote written by author name given below:

author:  

quote: 


In [13]:
df['prompt'] = df.apply(lambda x: f'{start_prompt}{x["author"]}{end_prompt}{x["quote"]}', axis=1)

In [14]:
print(df['full_text'].iloc[9])
print("################")
print(df['prompt'].iloc[9])

author: Mahatma Gandhi
quote:“Be the change that you wish to see in the world.”
################
Please generate a quote written by author name given below:

author: Mahatma Gandhi

quote: “Be the change that you wish to see in the world.”


We will be treating this in two different problem sets. First we will finetune model for text generation
Next we will do it in QA format like given the author below, generate a quote

In [16]:
df['input_ids_len'] = df['full_text'].apply(lambda x: len(tokenizer(x, truncation=True).input_ids))
df = df.sort_values(by=['input_ids_len'], ascending=True)
df        

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,quote,author,tags,full_text,prompt,input_ids_len
62,“,"Kent M. Keith,","[forgiveness, goodness, happiness, honesty, ki...","author: Kent M. Keith,\nquote:“",Please generate a quote written by author name...,13
1050,“Don't Panic.”,"Douglas Adams,","[h2g2, hitchhiker-s-guide, humor, panic]","author: Douglas Adams,\nquote:“Don't Panic.”",Please generate a quote written by author name...,17
2141,“Hope is a waking dream.”,Aristotle,"[dreams, hope]",author: Aristotle\nquote:“Hope is a waking dre...,Please generate a quote written by author name...,17
562,“I cannot live without books.”,Thomas Jefferson,[books],author: Thomas Jefferson\nquote:“I cannot live...,Please generate a quote written by author name...,18
327,“Peace begins with a smile..”,Mother Teresa,"[inspirational, peace, smile, smiling]",author: Mother Teresa\nquote:“Peace begins wit...,Please generate a quote written by author name...,18
...,...,...,...,...,...,...
411,“I can believe things that are true and things...,"Neil Gaiman,",[belief],"author: Neil Gaiman,\nquote:“I can believe thi...",Please generate a quote written by author name...,655
1878,"“Once upon a midnight dreary, while I pondered...","Edgar Allan Poe,","[hopelessness, pain]","author: Edgar Allan Poe,\nquote:“Once upon a m...",Please generate a quote written by author name...,819
1051,"“For me, trees have always been the most penet...","Herman Hesse,","[eternity, fear, forests, history, holiness, h...","author: Herman Hesse,\nquote:“For me, trees ha...",Please generate a quote written by author name...,838
2418,“I will love you as a thief loves a gallery an...,Lemony Snicket,[the-beatrice-letters],author: Lemony Snicket\nquote:“I will love you...,Please generate a quote written by author name...,867


In [17]:
df.describe()

,input_ids_len
count,2508.000000
mean,53.240431
std,61.133405
min,13.000000
25%,29.000000
50%,37.000000
75%,53.250000
max,965.000000


In [18]:
df['input_ids_len'].quantile(0.98)

213.0

In [19]:
df_1 = df[df['input_ids_len'] < 512].copy()
df_1.reset_index(inplace=True, drop=True)
df_1['id'] = range(len(df_1))

In [20]:
df_1

,quote,author,tags,full_text,prompt,input_ids_len,id
0,“,"Kent M. Keith,","[forgiveness, goodness, happiness, honesty, ki...","author: Kent M. Keith,\nquote:“",Please generate a quote written by author name...,13,0
1,“Don't Panic.”,"Douglas Adams,","[h2g2, hitchhiker-s-guide, humor, panic]","author: Douglas Adams,\nquote:“Don't Panic.”",Please generate a quote written by author name...,17,1
2,“Hope is a waking dream.”,Aristotle,"[dreams, hope]",author: Aristotle\nquote:“Hope is a waking dre...,Please generate a quote written by author name...,17,2
3,“I cannot live without books.”,Thomas Jefferson,[books],author: Thomas Jefferson\nquote:“I cannot live...,Please generate a quote written by author name...,18,3
4,“Peace begins with a smile..”,Mother Teresa,"[inspirational, peace, smile, smiling]",author: Mother Teresa\nquote:“Peace begins wit...,Please generate a quote written by author name...,18,4
...,...,...,...,...,...,...,...
2492,"“Tess, Tess, Tessa. Was there ever a more beau...","Cassandra Clare,","[cassandra-clare, clockwork-prince, tessa-gray...","author: Cassandra Clare,\nquote:“Tess, Tess, T...",Please generate a quote written by author name...,352,2492
2493,"“To be, or not to be: that is the question:Whe...","William Shakespeare,","[death, existence, life]","author: William Shakespeare,\nquote:“To be, or...",Please generate a quote written by author name...,377,2493
2494,“Ø¥Ù† Ø§Ù„Ù…Ø±Ø£Ø© ØªØ­Ø¨ Ø±Ø¬Ù„Ù‡Ø§ Ù„ÙŠØ³ Ù„...,Ø£Ø­Ù…Ø¯ Ø®Ø§Ù„Ø¯ ØªÙˆÙ�ÙŠÙ‚,[life],author: Ø£Ø­Ù…Ø¯ Ø®Ø§Ù„Ø¯ ØªÙˆÙ�ÙŠÙ‚\nquote:“Ø...,Please generate a quote written by author name...,455,2494
2495,“Look again at that dot. That's here. That's h...,"Carl Sagan,","[earth, perspective, space]","author: Carl Sagan,\nquote:“Look again at that...",Please generate a quote written by author name...,468,2495


In [21]:
df.to_csv('./quotes.csv', index=False)

In [22]:
tokenizer.pad_token, tokenizer.eos_token

('<pad>', '</s>')

In [23]:
!pip install torchinfo

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [24]:
from torchinfo import summary

In [25]:
model = AutoModelForCausalLM.from_pretrained('facebook/opt-350m')

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [27]:
summary(model)

Layer (type:depth-idx)                             Param #
OPTForCausalLM                                     --
├─OPTModel: 1-1                                    --
│    └─OPTDecoder: 2-1                             --
│    │    └─Embedding: 3-1                         25,739,264
│    │    └─OPTLearnedPositionalEmbedding: 3-2     2,099,200
│    │    └─Linear: 3-3                            524,288
│    │    └─Linear: 3-4                            524,288
│    │    └─ModuleList: 3-5                        302,309,376
├─Linear: 1-2                                      25,739,264
Total params: 356,935,680
Trainable params: 356,935,680
Non-trainable params: 0

In [28]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features

In [29]:
model.config

OPTConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "facebook/opt-350m",
  "_remove_final_layer_norm": false,
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "OPTForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "do_layer_norm_before": false,
  "dropout": 0.1,
  "enable_bias": true,
  "eos_token_id": 2,
  "ffn_dim": 4096,
  "hidden_size": 1024,
  "init_std": 0.02,
  "layer_norm_elementwise_affine": true,
  "layerdrop": 0.0,
  "max_position_embeddings": 2048,
  "model_type": "opt",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "prefix": "</s>",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "use_cache": true,
  "vocab_size": 50272,
  "word_embed_proj_dim": 512
}

In [30]:
tokenizer.decode([1,2,3])

'<pad></s><unk>'

In [32]:
vocab_size =  model.config.vocab_size
vocab_size

50272

In [33]:
tokenizer.encode(tokenizer.eos_token), tokenizer.encode(tokenizer.bos_token), tokenizer.encode(tokenizer.pad_token)

([2, 2], [2, 2], [2, 1])

##### First we wil fine tune the model in text generation model and next we will do it in QA mode

In [4]:
#Sagemaker related commands
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
sess, role, region, bucket

[03/06/25 08:11:36] INFO     Found credentials from IAM Role:                                   ]8;id=112658;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=967895;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=326508;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=400561;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

(<sagemaker.session.Session at 0x7fb5671ccd60>,
 'arn:aws:iam::191013407134:role/service-role/AmazonSageMaker-ExecutionRole-20250124T222384',
 'us-east-1',
 'sagemaker-us-east-1-191013407134')

In [5]:
%pwd

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/FineTuneDecoder'

In [6]:
S3_LOC = f's3://{bucket}/dataset/abirate_quotes/'
S3_LOC

's3://sagemaker-us-east-1-191013407134/dataset/abirate_quotes/'

In [7]:
#!aws s3 cp ./quotes.csv {S3_LOC}

In [8]:
!aws s3 ls --recursive {S3_LOC}

2025-03-05 12:57:56    4146715 dataset/abirate_quotes/model_data/model.tar.gz
2025-03-04 05:39:55    1763910 dataset/abirate_quotes/quotes.csv


In [9]:
# let us get image uri
training_image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch', 
    version='2.0',
    instance_type='ml.g4dn.xlarge',
    region=region,
    py_version='py310',
    image_scope='training'
)
print(training_image_uri)

763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310


In [10]:
train_uri = f'{S3_LOC}'
train_uri

's3://sagemaker-us-east-1-191013407134/dataset/abirate_quotes/'

In [11]:
s3_inp_tr = TrainingInput(
    s3_data = train_uri
    )
s3_inp_tr

In [12]:
data_channels = {
    'train': s3_inp_tr
    }
data_channels

{'train': <sagemaker.inputs.TrainingInput at 0x7fb56093a740>}

In [13]:
objective_metric_name = "average training loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "average training loss", "Regex": "Average loss: ([0-9\\.]+)"},
                      {"Name": "Perplexity", "Regex": "Perplexity: ([0-9\\.]+)"}]
objective_metric_name, objective_type, metric_definitions

('average training loss',
 'Minimize',
 [{'Name': 'average training loss', 'Regex': 'Average loss: ([0-9\\.]+)'},
  {'Name': 'Perplexity', 'Regex': 'Perplexity: ([0-9\\.]+)'}])

In [14]:
ic=1
i_type = "ml.g4dn.xlarge"
ic, i_type

(1, 'ml.g4dn.xlarge')

In [15]:
%pwd

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/FineTuneDecoder'

# let us get image uri
training_image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch', 
    version='2.0',
    instance_type='ml.g6.xlarge',
    region=region,
    py_version='py310',
    image_scope='training'
)
print(training_image_uri)

ic=1
i_type = "ml.g6.xlarge"
ic, i_type

In [81]:
hparams = {
    'bs': 16,
    'lrate': 0.0008,
    'num_epochs': 20,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 4,
    'peft': "lora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 16,
 'lrate': 0.0008,
 'num_epochs': 20,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 4,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [82]:
%pwd

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/FineTuneDecoder'

In [83]:
est_qoutes_textgen_4bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-quotes-textgen.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='quotes-text-gen-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_qoutes_textgen_4bit.fit(
    inputs=data_channels,
    wait=True
)

[03/04/25 11:03:49] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=536567;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=602602;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[03/04/25 11:03:50] INFO     Creating training-job with name:                                       ]8;id=659786;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=435271;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             quotes-text-gen-train-2025-03-04-11-03-49-977                                         

2025-03-04 11:03:50 Starting - Starting the training job......
..25-03-04 11:04:28 Starting - Preparing the instances for training.
....................Downloading - Downloading the training image.
bash: cannot set terminal process group (-1): Inappropriate ioctl for devices..
bash: no job control in this shell
2025-03-04 11:08:58,405 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-04 11:08:58,424 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-04 11:08:58,434 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-04 11:08:58,441 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-04 11:09:00,423 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.1 MB/s eta 0:00:00
━━

In [46]:
est_qoutes_textgen_4bit = Estimator.attach('quotes-text-gen-train-2025-03-04-11-03-49-977')

[03/05/25 05:09:08] INFO     Found credentials from IAM Role:                                   ]8;id=438435;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=888281;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

[03/05/25 05:09:09] INFO     Found credentials from IAM Role:                                   ]8;id=421767;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=448455;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   


2025-03-04 11:28:21 Starting - Preparing the instances for training
2025-03-04 11:28:21 Downloading - Downloading the training image
2025-03-04 11:28:21 Training - Training image download completed. Training in progress.
2025-03-04 11:28:21 Uploading - Uploading generated training model
2025-03-04 11:28:21 Completed - Training job completed


In [47]:
!aws s3 ls --recursive {est_qoutes_textgen_4bit.model_data}
!aws s3 cp {est_qoutes_textgen_4bit.model_data} ./est_qoutes_textgen_4bit/
%cd est_qoutes_textgen_4bit
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh


%cd ..
%pwd


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2025-03-04 11:28:14    4148146 quotes-text-gen-train-2025-03-04-11-03-49-977/output/model.tar.gz


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


download: s3://sagemaker-us-east-1-191013407134/quotes-text-gen-train-2025-03-04-11-03-49-977/output/model.tar.gz to est_qoutes_textgen_4bit/model.tar.gz
/home/ec2-user/SageMaker/DecoderTasks/est_qoutes_textgen_4bit
total 4.0M
-rw-rw-r-- 1 ec2-user ec2-user 4.0M Mar  4 11:28 model.tar.gz


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/adapter_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_4.906924388591055/tokenizer_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_20_Prplxty_

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


'/home/ec2-user/SageMaker/DecoderTasks'

In [48]:
%pwd

'/home/ec2-user/SageMaker/DecoderTasks'

In [49]:
%cd est_qoutes_textgen_4bit/ckpt_Epoch_20_Prplxty_4.906924388591055/
%pwd

/home/ec2-user/SageMaker/DecoderTasks/est_qoutes_textgen_4bit/ckpt_Epoch_20_Prplxty_4.906924388591055


'/home/ec2-user/SageMaker/DecoderTasks/est_qoutes_textgen_4bit/ckpt_Epoch_20_Prplxty_4.906924388591055'

In [50]:
!ls -ltrh

total 7.7M
-rw-r--r-- 1 ec2-user ec2-user 5.0K Mar  4 11:27 README.md
-rw-r--r-- 1 ec2-user ec2-user  700 Mar  4 11:27 tokenizer_config.json
-rw-r--r-- 1 ec2-user ec2-user  548 Mar  4 11:27 special_tokens_map.json
-rw-r--r-- 1 ec2-user ec2-user 3.1M Mar  4 11:27 adapter_model.safetensors
-rw-r--r-- 1 ec2-user ec2-user  712 Mar  4 11:27 adapter_config.json
-rw-r--r-- 1 ec2-user ec2-user 780K Mar  4 11:27 vocab.json
-rw-r--r-- 1 ec2-user ec2-user 446K Mar  4 11:27 merges.txt
-rw-r--r-- 1 ec2-user ec2-user 3.4M Mar  4 11:27 tokenizer.json
-rw-r--r-- 1 ec2-user ec2-user 5.2K Mar  4 11:27 training_args.bin


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [52]:
from peft import AutoPeftModelForCausalLM

In [53]:
loaded_model = AutoPeftModelForCausalLM.from_pretrained('./', is_trainable=False)

In [54]:
loaded_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50265, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=8, bias=Fal

In [55]:
summary(loaded_model)

Layer (type:depth-idx)                                                 Param #
PeftModelForCausalLM                                                   --
├─LoraModel: 1-1                                                       --
│    └─OPTForCausalLM: 2-1                                             --
│    │    └─OPTModel: 3-1                                              (331,979,264)
│    │    └─Linear: 3-2                                                (25,735,680)
Total params: 357,714,944
Trainable params: 0
Non-trainable params: 357,714,944

In [144]:
def generate(prompt):
    tokenized_text = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = loaded_model.generate(**tokenized_text,
                                   do_sample=True,
                                   top_k=3,
                                   temperature=0.8,
                                   max_new_tokens=50)
    result = tokenizer.batch_decode(output,  skip_special_tokens=True)[0]
    return result

In [145]:
full_text = '''author: Mahatma Gandhi
quote:"Be the change that you wish to see in the world."
'''

print(full_text)

author: Mahatma Gandhi
quote:"Be the change that you wish to see in the world."



In [146]:
prompt = '''author: Mahatma Gandhi
quote:"Be the change '''
print(prompt)

author: Mahatma Gandhi
quote:"Be the change 


In [149]:
generate(prompt)

'author: Mahatma Gandhi\nquote:"Be the change  you want to see in the world. But don\'t make yourself a part of it. Be a part of your own change. Be a part of the transformation that is coming. But don\'t be a part of the process that is coming for you'

In [137]:
#So the model is generating correct text and some blabber lol!

In [150]:
prompt = '''author: Mahatma Gandhi
quote:"'''
print(prompt)
print(generate(prompt))

author: Mahatma Gandhi
quote:"
author: Mahatma Gandhi
quote:"I do not want to be the only one who has seen the world. I am sure that there are many who have not, and will never see it. But I want to be able to give a little of my wisdom to those who have not


In [141]:
#when we let model choose it wanders around..expected

In [155]:
prompt = '''author: Mahatma Gandhi
quote:"Nobody can hurt'''
print(prompt)
print(generate(prompt))

author: Mahatma Gandhi
quote:"Nobody can hurt
author: Mahatma Gandhi
quote:"Nobody can hurt me without my consent. I have to give up my own power to help others."


In [156]:
prompt = '''author: Mahatma Gandhi
quote:"An eye for an eye'''
print(prompt)
print(generate(prompt))

author: Mahatma Gandhi
quote:"An eye for an eye
author: Mahatma Gandhi
quote:"An eye for an eye will make the whole world blind, but an eye for an eye will make everything light.”


In [182]:
prompt = '''author: Mahatma Gandhi
quote:"Each night, when I go to sleep'''
print(prompt)
print(generate(prompt))

author: Mahatma Gandhi
quote:"Each night, when I go to sleep
author: Mahatma Gandhi
quote:"Each night, when I go to sleep, I tell myself that the universe is a complex system of which I have no control. But, I am not alone. I am not alone. I am here. And I will continue to live here. And I will continue to be here.


In [89]:
hparams = {
    'bs': 16,
    'lrate': 0.0009,
    'num_epochs': 30,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 0,
    'peft': "lora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 16,
 'lrate': 0.0009,
 'num_epochs': 30,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 0,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [90]:
est_qoutes_textgen_loraonly = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-quotes-textgen.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='quotes-text-gen-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_qoutes_textgen_loraonly.fit(
    inputs=data_channels,
    wait=True
)

[03/04/25 11:44:35] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=529250;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=690276;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=473683;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=817213;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             quotes-text-gen-train-2025-03-04-11-44-35-554                                         

2025-03-04 11:44:36 Starting - Starting the training job......
..25-03-04 11:45:12 Starting - Preparing the instances for training.
..25-03-04 11:45:57 Downloading - Downloading input data.
....................Downloading - Downloading the training image.
2025-03-04 11:49:44 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2025-03-04 11:49:59,416 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-04 11:49:59,435 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-04 11:49:59,446 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-04 11:49:59,452 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-04 11:50:01,418 sagemaker-training-toolkit INFO     Installing dependencies from requirements.t

In [183]:
est_qoutes_textgen_loraonly = Estimator.attach('quotes-text-gen-train-2025-03-04-11-44-35-554')


2025-03-04 12:15:33 Starting - Preparing the instances for training
2025-03-04 12:15:33 Downloading - Downloading the training image
2025-03-04 12:15:33 Training - Training image download completed. Training in progress.
2025-03-04 12:15:33 Uploading - Uploading generated training model
2025-03-04 12:15:33 Completed - Training job completed


In [185]:
%cd ../..
%pwd

/home/ec2-user/SageMaker/DecoderTasks


'/home/ec2-user/SageMaker/DecoderTasks'

In [186]:
!aws s3 ls --recursive {est_qoutes_textgen_loraonly.model_data}
!aws s3 cp {est_qoutes_textgen_loraonly.model_data} ./est_qoutes_textgen_loraonly/
%cd est_qoutes_textgen_loraonly
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh

%cd ..
%pwd


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


2025-03-04 12:15:26    4146715 quotes-text-gen-train-2025-03-04-11-44-35-554/output/model.tar.gz


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


download: s3://sagemaker-us-east-1-191013407134/quotes-text-gen-train-2025-03-04-11-44-35-554/output/model.tar.gz to est_qoutes_textgen_loraonly/model.tar.gz
/home/ec2-user/SageMaker/DecoderTasks/est_qoutes_textgen_loraonly
total 4.0M
-rw-rw-r-- 1 ec2-user ec2-user 4.0M Mar  4 12:15 model.tar.gz


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/merges.txt
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/README.md
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/adapter_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/tokenizer.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_30_Prplxty_3.6311811785448933/sp

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


'/home/ec2-user/SageMaker/DecoderTasks'

In [188]:
loaded_model_peftonly = AutoPeftModelForCausalLM.from_pretrained('./est_qoutes_textgen_loraonly/ckpt_Epoch_30_Prplxty_3.6311811785448933', is_trainable=False)

In [189]:
loaded_model_peftonly

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50265, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=8, bias=Fal

In [190]:
summary(loaded_model_peftonly)

Layer (type:depth-idx)                                                 Param #
PeftModelForCausalLM                                                   --
├─LoraModel: 1-1                                                       --
│    └─OPTForCausalLM: 2-1                                             --
│    │    └─OPTModel: 3-1                                              (331,979,264)
│    │    └─Linear: 3-2                                                (25,735,680)
Total params: 357,714,944
Trainable params: 0
Non-trainable params: 357,714,944

In [195]:
loaded_model_peftonly.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50265, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=8, bias=Fal

In [216]:
def generate(prompt, loaded_model):
    tokenized_text = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = loaded_model.generate(**tokenized_text,
                                   do_sample=True,
                                   top_k=3,
                                   temperature=0.9,
                                   max_new_tokens=30)
    result = tokenizer.batch_decode(output,  skip_special_tokens=True)[0]
    return result

In [217]:
prompt = '''author: Mahatma Gandhi
quote:"Each night, when I go to sleep'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mahatma Gandhi
quote:"Each night, when I go to sleep
author: Mahatma Gandhi
quote:"Each night, when I go to sleep, I sleep for only a second. Then I wake up and it's all over. The other night, when I woke up, it was just


In [218]:
prompt = '''author: Mahatma Gandhi
quote:"Be the change'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mahatma Gandhi
quote:"Be the change
author: Mahatma Gandhi
quote:"Be the change that you want to see in the world. And don't expect people to give up anything. They have worked hard for it. If you can't


In [219]:
prompt = '''author: Mahatma Gandhi
quote:"Nobody can hurt'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mahatma Gandhi
quote:"Nobody can hurt
author: Mahatma Gandhi
quote:"Nobody can hurt me without my permission. I will not be hurt by anybody other than my own self. This is the way it has been for thousands of years.


In [228]:
prompt = '''author: Mahatma Gandhi
quote:"Freedom is not worth'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mahatma Gandhi
quote:"Freedom is not worth
author: Mahatma Gandhi
quote:"Freedom is not worth having if it doesn't include the ability to be treated with dignity and respect."


In [229]:
prompt = '''author: Mahatma Gandhi
quote:"Hate the sin'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mahatma Gandhi
quote:"Hate the sin
author: Mahatma Gandhi
quote:"Hate the sin, love the sinner. This is why, when a child is found in an open field, they say, "I was here before you."


In [230]:
#it seems it does better when the target quote is short

In [240]:
prompt = '''author: Mark Twain
quote:"Sanity and happiness'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mark Twain
quote:"Sanity and happiness
author: Mark Twain
quote:"Sanity and happiness are not the only possibilities, but they are the only two possibilities that exist.”


In [247]:
prompt = '''author: Mark Twain
quote:"Kindness is a language'''
print(prompt)
print(generate(prompt, loaded_model_peftonly))

author: Mark Twain
quote:"Kindness is a language
author: Mark Twain
quote:"Kindness is a language that can be heard, seen, and felt in the hearts of many peoples. It is a language that can be heard, seen, and felt in


In [248]:
loaded_model_peftonly_test = AutoPeftModelForCausalLM.from_pretrained('./est_qoutes_textgen_loraonly/ckpt_Epoch_30_Prplxty_3.6311811785448933', is_trainable=True)

In [249]:
summary(loaded_model_peftonly_test)

Layer (type:depth-idx)                                                 Param #
PeftModelForCausalLM                                                   --
├─LoraModel: 1-1                                                       --
│    └─OPTForCausalLM: 2-1                                             --
│    │    └─OPTModel: 3-1                                              331,979,264
│    │    └─Linear: 3-2                                                (25,735,680)
Total params: 357,714,944
Trainable params: 786,432
Non-trainable params: 356,928,512

In [261]:
prompt = 'Be the change'
print(prompt)
print(generate(prompt, loaded_model_peftonly))

Be the change
Be the change you want to see in the world.

I don't want to be a good woman. I want to be the kind who can make the


### Prompt Tuning

In [278]:
model_name = "bigscience/bloomz-560m"

In [279]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [280]:
input1 = tokenizer("Two things are infinite: ", return_tensors="pt")

foundation_outputs = foundation_model.generate(
    input_ids=input1["input_ids"], 
    attention_mask=input1["attention_mask"], 
    max_new_tokens=7, 
    eos_token_id=tokenizer.eos_token_id
    )
print(tokenizer.batch_decode(foundation_outputs, skip_special_tokens=True))

['Two things are infinite:  the number of people and the number']


In [281]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")

data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)
train_sample = data["train"].select(range(1500))
display(train_sample) 

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 1500
})

In [282]:
from peft import  get_peft_model, PromptTuningConfig, TaskType, PromptTuningInit

peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=6,
    tokenizer_name_or_path=model_name
)
peft_model = get_peft_model(foundation_model, peft_config)
print(peft_model.print_trainable_parameters())

trainable params: 6,144 || all params: 559,220,736 || trainable%: 0.0011
None


In [283]:
summary(peft_model)

Layer (type:depth-idx)                             Param #
PeftModelForCausalLM                               --
├─BloomForCausalLM: 1-1                            --
│    └─BloomModel: 2-1                             --
│    │    └─Embedding: 3-1                         (256,901,120)
│    │    └─LayerNorm: 3-2                         (2,048)
│    │    └─ModuleList: 3-3                        (302,309,376)
│    │    └─LayerNorm: 3-4                         (2,048)
│    └─Linear: 2-2                                 (256,901,120)
├─ModuleDict: 1-2                                  --
│    └─PromptEmbedding: 2-3                        --
│    │    └─Embedding: 3-5                         6,144
├─Embedding: 1-3                                   (recursive)
Total params: 816,121,856
Trainable params: 6,144
Non-trainable params: 816,115,712

In [284]:
from transformers import TrainingArguments
import os

output_directory = os.path.join("./peft_outputs")

os.makedirs(output_directory, exist_ok=True)

In [285]:
training_args = TrainingArguments(
    output_dir=output_directory, # Where the model predictions and checkpoints will be written
    auto_find_batch_size=True, # Find a suitable batch size that will fit into memory automatically 
    learning_rate= 3e-2, # Higher learning rate than full fine-tuning
    num_train_epochs=20 # Number of passes to go through the entire fine-tuning dataset 
)

In [286]:
from transformers import Trainer, DataCollatorForLanguageModeling

In [287]:
trainer = Trainer(
    model=peft_model, # We pass in the PEFT version of the foundation model, bloomz-560M
    args=training_args,
    train_dataset=train_sample,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False) # mlm=False indicates not to use masked language modeling
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [288]:
trainer.train()

Step,Training Loss


Step,Training Loss


Step,Training Loss


Step,Training Loss
500,3.214500
1000,3.152200
1500,3.153900
2000,3.120100
2500,3.148000
3000,3.121300
3500,3.076600
4000,3.091800
4500,3.179500
5000,3.085200


TrainOutput(global_step=30000, training_loss=3.083267822265625, metrics={'train_runtime': 1854.7488, 'train_samples_per_second': 16.175, 'train_steps_per_second': 16.175, 'total_flos': 2151980963561472.0, 'train_loss': 3.083267822265625, 'epoch': 20.0})

In [289]:
import time

time_now = time.time()
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

In [292]:
!ls -ltrh {peft_model_path}

total 40K
-rw-rw-r-- 1 ec2-user ec2-user 5.0K Mar  5 07:06 README.md
-rw-rw-r-- 1 ec2-user ec2-user  25K Mar  5 07:06 adapter_model.safetensors
-rw-rw-r-- 1 ec2-user ec2-user  470 Mar  5 07:06 adapter_config.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [293]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(foundation_model.to(device), 
                                         peft_model_path, 
                                         is_trainable=False)


In [310]:
loaded_model_outputs = loaded_model.generate(
    input_ids=input1["input_ids"].to(device), 
    attention_mask=input1["attention_mask"].to(device), 
    max_new_tokens=7, 
    eos_token_id=tokenizer.eos_token_id,
    #temperature=0.8,
    #do_sample=True,
    #top_k=3
    )
print(tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True))


['Two things are infinite:  the number of people who love you']


##### Retraining LORA Only peft model that we fine tuned for text generation as supeervised fine tuning task .. kind of QA task

In [43]:
hparams = {
    'bs': 8,
    'lrate': 0.0009,
    'num_epochs': 30,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 0,
    'peft': "peft_retrain",
    'grad_accum_steps': 1 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0009,
 'num_epochs': 30,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 0,
 'peft': 'peft_retrain',
 'grad_accum_steps': 1}

In [44]:
est_qoutes_QA_loraretrain = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-quotes-textgen-QA.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='quotes-text-gen-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_qoutes_QA_loraretrain.fit(
    inputs=data_channels,
    wait=True
)

[03/06/25 09:15:34] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=911180;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=304821;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=86716;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=597608;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             quotes-text-gen-train-2025-03-06-09-15-34-558                                         

2025-03-06 09:15:35 Starting - Starting the training job...
..25-03-06 09:15:55 Starting - Preparing the instances for training.
..25-03-06 09:16:23 Downloading - Downloading input data.
....................Downloading - Downloading the training image.
.bash: cannot set terminal process group (-1): Inappropriate ioctl for device..
bash: no job control in this shell
2025-03-06 09:20:49,885 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-06 09:20:49,905 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-06 09:20:49,916 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-06 09:20:49,923 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-06 09:20:51,850 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━